# Section 05g Onboarding Notebook: Many-Body Green's Function Methods — Introduction to Green's Functions, $GW$, Bethe–Salpeter Equation (`BSE`), and Dynamical Mean-Field Theory (`DMFT`)

### Overview & Pedagogical Design
While wavefunction methods (`CI`, `CC`, `CASSCF`) focus on the $N$-electron ground-state wavefunction $|\Psi_0^N\rangle$, spectroscopic experiments probe **excitations** of the many-body system:
- **Photoemission & Inverse Photoemission Spectroscopy (PES / IPES / ARPES)** measure **charged excitations** ($N \to N \pm 1$ electrons: Ionization Potentials $\text{IP}$ and Electron Affinities $\text{EA}$, which define the fundamental transport gap $E_{\text{gap}}^{\text{QP}} = \text{IP} - \text{EA}$). These are the exact poles of the **One-Particle Green's Function $G(\omega)$**, computed via **Dyson's Equation**, **Second-Order Green's Function (`GF2`)** theory, or Hedin's **$GW$ Approximation** ([PySCF GW/PBC Guide](https://pyscf.org/user/pbc/scf.html)).
- **Optical Absorption & UV-Vis Spectroscopy** measure **neutral excitations** ($N \to N$ electrons, where a promoted electron and the hole it leaves behind attract one another via the screened Coulomb interaction to form a bound **exciton**). These are the poles of the **Two-Particle (Electron–Hole) Propagator $L(\omega)$**, governed by the **Bethe–Salpeter Equation (`BSE`)**.
- **Strongly Correlated Mott Insulators & Kondo Systems** (where local Coulomb repulsion $U$ is comparable to or larger than the bandwidth $W$, causing perturbative $GW$ to break down) are governed by **Dynamical Mean-Field Theory (`DMFT`)**, which maps the correlated lattice onto a self-consistent quantum impurity embedded in a dynamical hybridization bath $\Delta(\omega)$.

In this notebook, **no Green's function, $GW$, `BSE`, or `DMFT` functions are pre-solved for you**. You will implement **16 granular `# TODO` functions** walking step-by-step through all four frameworks:
- **Part 1 — Introduction to Green's Functions: $G_0(\omega)$, Spectral Functions $A(\omega)$, Dyson's Equation, `GF2` Self-Energy & Quasiparticle Weights $Z_p$ (`TODO 1` – `TODO 4`)**
- **Part 2 — Hedin's $GW$ Approximation: Direct Random Phase Approximation (`dRPA`) Screening, Transition Potentials $W_{pq}^m$ & $G_0W_0$ Quasiparticle Solver (`TODO 5` – `TODO 8`)**
- **Part 3 — Bethe–Salpeter Equation (`BSE`): Static Screened Coulomb Kernel $W_{pq,rs}(0)$, Exciton Hamiltonian, Exciton Binding Energy & Optical Spectrum (`TODO 9` – `TODO 12`)**
- **Part 4 — Dynamical Mean-Field Theory (`DMFT`): Bethe Lattice Hilbert Transform, Self-Consistent Impurity Loop, Three-Peak Kondo/Hubbard Spectrum & Mott Transition (`TODO 13` – `TODO 16`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section05g_greens_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh, sqrtm
from pyscf import gto, scf
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
HARTREE_TO_EV = 27.211386245988

# Build and solve the RHF reference directly in PySCF for H2O (STO-3G, 10 electrons, 7 MOs)
mol_h2o = gto.M(
    atom="O 0.0 0.0 0.0; H 0.0 -0.757 0.587; H 0.0 0.757 0.587",
    basis="sto-3g",
    verbose=0,
)
mf_h2o = scf.RHF(mol_h2o).run(conv_tol=1e-12)
eps_hf_h2o = mf_h2o.mo_energy
C_hf_h2o = mf_h2o.mo_coeff
eri_ao_h2o = mol_h2o.intor("int2e")
eri_mo_h2o = np.einsum("uvls,up,vq,lr,st->pqrt", eri_ao_h2o, C_hf_h2o, C_hf_h2o, C_hf_h2o, C_hf_h2o, optimize=True)
with mol_h2o.with_common_orig((0.0, 0.0, 0.0)):
    ao_dip_h2o = mol_h2o.intor_symmetric("int1e_r", comp=3)

print(f"Initialized PySCF RHF Reference for H2O (STO-3G): E_RHF = {mf_h2o.e_tot:.10f} Eh")
print("Implement TODOs 1-16 and run `run_section05g_greens_checkpoints()`!")

## Part 1: Introduction to Many-Body Green's Functions — Lehmann Representation, Dyson's Equation & `GF2` (TODOs 1–4)

### Mathematical Specifications
1. **Non-Interacting Retarded Green's Function $G_0^R(\omega)$ & Spectral Function $A_0(\omega)$ (`evaluate_noninteracting_greens_function` — TODO 1)**:
   In the canonical Hartree–Fock MO basis $\{\phi_p\}$ with eigenvalues $\{\varepsilon_p\}$ and Lorentzian broadening $\eta > 0$, the retarded non-interacting Green's function matrix is diagonal:
   $$G_{pq}^{R,(0)}(\omega) = \frac{\delta_{pq}}{\omega - \varepsilon_p + i\eta}, \qquad A_0(\omega) = -\frac{1}{\pi}\,\text{Im}\,\text{Tr}\,\mathbf{G}^{R,(0)}(\omega) = \sum_{p=1}^{N_{\text{MO}}} \frac{1}{\pi}\frac{\eta}{(\omega - \varepsilon_p)^2 + \eta^2}$$
   Notice that each MO contributes unit spectral weight $\int_{-\infty}^{+\infty} A_{0,p}(\omega)\,d\omega = 1$, so $\int_{-\infty}^{+\infty} A_0(\omega)\,d\omega = N_{\text{MO}}$.

2. **Dyson's Equation in Matrix Form (`solve_dyson_equation` — TODO 2)**:
   Given the diagonal HF Fock matrix $\mathbf{F}_{\text{HF}} = \text{diag}(\boldsymbol{\varepsilon})$ and a correlation self-energy matrix $\boldsymbol{\Sigma}_c(\omega)$ at frequency $\omega$:
   $$\mathbf{G}^R(\omega) = \Big[(\omega + i\eta)\mathbf{I} - \mathbf{F}_{\text{HF}} - \boldsymbol{\Sigma}_c(\omega)\Big]^{-1}, \qquad A(\omega) = -\frac{1}{\pi}\,\text{Im}\,\text{Tr}\,\mathbf{G}^R(\omega)$$

3. **Second-Order Green's Function (`GF2`) Correlation Self-Energy (`evaluate_gf2_self_energy_diagonal` — TODO 3)**:
   For closed-shell spatial orbitals ($i,j \in \{0, \dots, n_{\text{occ}}-1\}$ and $a,b \in \{n_{\text{occ}}, \dots, N_{\text{MO}}-1\}$), the diagonal retarded `GF2` correlation self-energy $\Sigma_{pp}^{(2)}(\omega)$ consists of a **2-particle-1-hole (2p1h)** forward polarization term and a **2-hole-1-particle (2h1p)** backward relaxation term:
   $$\Sigma_{pp}^{(2)}(\omega) = \underbrace{\sum_{i \in \text{occ}}\sum_{a,b \in \text{virt}} \frac{(pa|ib)\big[2(pa|ib) - (pb|ia)\big]}{\omega + \varepsilon_i - \varepsilon_a - \varepsilon_b + i\eta}}_{\Sigma_{pp}^{\text{2p1h}}(\omega)} + \underbrace{\sum_{i,j \in \text{occ}}\sum_{a \in \text{virt}} \frac{(pi|ja)\big[2(pi|ja) - (pa|ji)\big]}{\omega + \varepsilon_a - \varepsilon_i - \varepsilon_j + i\eta}}_{\Sigma_{pp}^{\text{2h1p}}(\omega)}$$

4. **Quasiparticle Renormalization Factor $Z_p$ & Linearized Quasiparticle Energy (`solve_linearized_quasiparticle_equation` — TODO 4)**:
   Given a callable `sigma_c_diag_fn(omega, p)` returning the complex scalar $\Sigma_{pp}^c(\omega)$, evaluate its numerical derivative $\frac{\partial\,\text{Re}\,\Sigma_{pp}^c(\omega)}{\partial \omega}\big|_{\omega = \varepsilon_p^{\text{HF}}}$ via central finite differences ($\pm d\omega$) to compute the **quasiparticle weight (residue)** $Z_p \in (0, 1]$ and linearized quasiparticle energy:
   $$Z_p = \left(1 - \frac{\partial\,\text{Re}\,\Sigma_{pp}^c(\omega)}{\partial \omega}\bigg|_{\omega = \varepsilon_p^{\text{HF}}}\right)^{-1}, \qquad \varepsilon_p^{\text{QP,lin}} = \varepsilon_p^{\text{HF}} + Z_p\,\text{Re}\,\Sigma_{pp}^c(\varepsilon_p^{\text{HF}})$$

In [ ]:
def evaluate_noninteracting_greens_function(
    omega_grid: np.ndarray, eps_hf: np.ndarray, eta: float = 0.05
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 1: Evaluate the diagonal retarded non-interacting Green's function G0_diag(omega, p)
    of shape (len(omega_grid), len(eps_hf)) and the total spectral function A0(omega) = -(1/pi) * sum_p Im G0_diag(omega, p)
    of shape (len(omega_grid),). Return (G0_diag, A0_omega).
    '''
    # TODO 1: Broadcast omega_grid[:, None] - eps_hf[None, :] + 1j * eta to form G0_diag and A0_omega.
    raise NotImplementedError("Complete TODO 1: evaluate_noninteracting_greens_function")


def solve_dyson_equation(
    omega: float, eps_hf: np.ndarray, Sigma_c_omega: np.ndarray, eta: float = 0.02
) -> tuple[np.ndarray, float]:
    '''
    TODO 2: Solve Dyson's matrix equation G(omega) = [(omega + i*eta)*I - diag(eps_hf) - Sigma_c_omega]^(-1)
    at scalar frequency `omega` and return (G_mat, spectral_trace_A), where
    spectral_trace_A = -(1/pi) * Im(Tr(G_mat)).
    '''
    # TODO 2: Invert the matrix using np.linalg.inv and compute -(1.0 / np.pi) * np.trace(G_mat).imag.
    raise NotImplementedError("Complete TODO 2: solve_dyson_equation")


def evaluate_gf2_self_energy_diagonal(
    omega: float, p_idx: int, eri_mo: np.ndarray, eps_hf: np.ndarray, n_occ: int, eta: float = 1e-4
) -> complex:
    '''
    TODO 3: Compute the retarded second-order Green's function (GF2) self-energy Sigma_{pp}^{(2)}(omega)
    for spatial orbital `p_idx` at frequency `omega` (returning a complex scalar).
    '''
    # TODO 3: Slice the 2p1h integrals (p, virt, occ, virt) and 2h1p integrals (p, occ, occ, virt) from eri_mo,
    #         build their frequency denominators with +1j * eta, and sum both terms.
    raise NotImplementedError("Complete TODO 3: evaluate_gf2_self_energy_diagonal")


def solve_linearized_quasiparticle_equation(
    p_idx: int, eps_hf: np.ndarray, sigma_c_fn, d_omega: float = 1e-4
) -> dict[str, float]:
    '''
    TODO 4: Using `sigma_c_fn(omega, p_idx) -> complex`, evaluate Re Sigma_c(eps_p) and its derivative
    d(Re Sigma_c)/d(omega) at omega = eps_hf[p_idx] via central difference (+/- d_omega).
    Return a dict with keys 'eps_hf', 're_sigma_c', 'd_sigma_domega', 'z_weight', 'eps_qp_lin'.
    '''
    # TODO 4: Evaluate sigma_c_fn at e0, e0 + d_omega, and e0 - d_omega, compute z_weight = 1 / (1 - d_sigma_domega),
    #         and eps_qp_lin = e0 + z_weight * re_sigma_c.
    raise NotImplementedError("Complete TODO 4: solve_linearized_quasiparticle_equation")

## Part 2: Hedin's $GW$ Approximation — Direct Random Phase Approximation (`dRPA`) & $G_0W_0$ Self-Energy (TODOs 5–8)

### Mathematical Specifications
1. **Direct Random Phase Approximation (`dRPA`) Casida Eigenvalue Problem (`build_and_solve_drpa` — TODO 5)**:
   In the occupied–virtual pair basis $|ia\rangle$ ($i \in \{0, \dots, n_{\text{occ}}-1\}$, $a \in \{n_{\text{occ}}, \dots, N_{\text{MO}}-1\}$, composite index $I = i \cdot n_{\text{virt}} + (a - n_{\text{occ}})$ of dimension $N_{ov} = n_{\text{occ}} n_{\text{virt}}$), the singlet **direct RPA (`dRPA`)** matrices (retaining only ring/bubble Coulomb diagrams without exchange) are:
   $$A_{ia, jb}^{\text{dRPA}} = (\varepsilon_a - \varepsilon_i)\delta_{ij}\delta_{ab} + 2(ia|jb), \qquad B_{ia, jb}^{\text{dRPA}} = 2(ia|jb)$$
   Note that $\mathbf{A}^{\text{dRPA}} - \mathbf{B}^{\text{dRPA}} = \text{diag}(\Delta\varepsilon_{ia})$ where $\Delta\varepsilon_{ia} = \varepsilon_a - \varepsilon_i > 0$. Construct the symmetric positive-definite matrix:
   $$M_{ia, jb}^{\text{dRPA}} = \sqrt{\Delta\varepsilon_{ia}}\,\left(A_{ia, jb}^{\text{dRPA}} + B_{ia, jb}^{\text{dRPA}}\right)\,\sqrt{\Delta\varepsilon_{jb}} = (\Delta\varepsilon_{ia})^2\delta_{ij}\delta_{ab} + 4\sqrt{\Delta\varepsilon_{ia}}\,(ia|jb)\,\sqrt{\Delta\varepsilon_{jb}}$$
   Diagonalizing $\mathbf{M}^{\text{dRPA}}\mathbf{Z}_m = (\Omega_m^{\text{RPA}})^2 \mathbf{Z}_m$ with `scipy.linalg.eigh` yields the exact RPA neutral charge-oscillation frequencies $\Omega_m^{\text{RPA}} = \sqrt{\lambda_m} > 0$ and the normalized transition density amplitudes:
   $$(X + Y)_{ia}^m = \sqrt{\frac{\Delta\varepsilon_{ia}}{\Omega_m^{\text{RPA}}}}\,Z_{ia, m}$$

2. **Screened 3-Index Transition Potentials $W_{pq}^m$ (`compute_rpa_transition_potentials` — TODO 6)**:
   Contract the $(N_{\text{MO}}, N_{\text{MO}}, n_{\text{occ}}, n_{\text{virt}})$ integral block $(pq|ia)$ with $(X + Y)_{ia}^m$ to obtain the 3-index screened transition potentials of shape `(N_ov, N_MO, N_MO)`:
   $$W_{pq}^m = \sum_{i \in \text{occ}}\sum_{a \in \text{virt}} (pq|ia)\,(X + Y)_{ia}^m$$

3. **Analytical Frequency Convolution for the $G_0W_0$ Correlation Self-Energy (`evaluate_g0w0_self_energy_diagonal` — TODO 7)**:
   Because our starting reference is Hartree–Fock ($\varepsilon_p^{\text{HF}}$ already includes the exact bare exchange self-energy $\Sigma_x = -K$), we only need the dynamical correlation part $\Sigma_{pp}^{c,GW}(\omega) = i G_0 * W^c$:
   $$\Sigma_{pp}^{c,GW}(\omega) = \sum_{m=1}^{N_{ov}} \left[ \sum_{i=0}^{n_{\text{occ}}-1} \frac{2\,\big|W_{pi}^m\big|^2}{\omega - \varepsilon_i^{\text{HF}} + \Omega_m^{\text{RPA}} - i\eta} + \sum_{a=n_{\text{occ}}}^{N_{\text{MO}}-1} \frac{2\,\big|W_{pa}^m\big|^2}{\omega - \varepsilon_a^{\text{HF}} - \Omega_m^{\text{RPA}} + i\eta} \right]$$

4. **Full $G_0W_0$ Quasiparticle Spectrum (`run_g0w0_quasiparticle_solver` — TODO 8)**:
   Compute the $G_0W_0$ quasiparticle energies $\varepsilon_p^{G_0W_0}$ and renormalization weights $Z_p^{GW}$ for all orbitals $p = 0, \dots, N_{\text{MO}}-1$, and compare the $G_0W_0$ fundamental gap $\Delta E_{\text{QP}}^{G_0W_0} = \varepsilon_{\text{LUMO}}^{G_0W_0} - \varepsilon_{\text{HOMO}}^{G_0W_0}$ against the bare Hartree–Fock HOMO–LUMO gap!

In [ ]:
def build_and_solve_drpa(
    eri_mo: np.ndarray, eps_hf: np.ndarray, n_occ: int
) -> dict[str, np.ndarray]:
    '''
    TODO 5: Build and solve the singlet direct RPA (dRPA) eigenvalue problem.
    Return a dict with keys:
        'omega_rpa': 1D array of shape (n_ov,) of positive RPA excitation energies
        'x_plus_y': 3D array of shape (n_occ, n_virt, n_ov) of transition densities (X + Y)_{ia}^m
        'A_drpa': 2D array of shape (n_ov, n_ov)
        'B_drpa': 2D array of shape (n_ov, n_ov)
    '''
    # TODO 5: Slice iajb = eri_mo[:n_occ, n_occ:, :n_occ, n_occ:], form delta_eps (n_ov,),
    #         build M_drpa, diagonalize with eigh, and compute omega_rpa = sqrt(evals) and x_plus_y.
    raise NotImplementedError("Complete TODO 5: build_and_solve_drpa")


def compute_rpa_transition_potentials(
    eri_mo: np.ndarray, x_plus_y: np.ndarray, n_occ: int
) -> np.ndarray:
    '''
    TODO 6: Contract eri_mo[:, :, :n_occ, n_occ:] (shape n_mo, n_mo, n_occ, n_virt)
    with `x_plus_y` (shape n_occ, n_virt, n_ov) to return `W_m_pq` of shape (n_ov, n_mo, n_mo).
    '''
    # TODO 6: Use np.einsum("pqia,iam->mpq", eri_mo[:, :, :n_occ, n_occ:], x_plus_y, optimize=True).
    raise NotImplementedError("Complete TODO 6: compute_rpa_transition_potentials")


def evaluate_g0w0_self_energy_diagonal(
    omega: float,
    p_idx: int,
    eps_hf: np.ndarray,
    omega_rpa: np.ndarray,
    W_m_pq: np.ndarray,
    n_occ: int,
    eta: float = 1e-4,
) -> complex:
    '''
    TODO 7: Evaluate the G0W0 correlation self-energy Sigma_{pp}^{c,GW}(omega) at frequency `omega`
    for orbital `p_idx` using `omega_rpa` (n_ov,) and `W_m_pq` (n_ov, n_mo, n_mo).
    '''
    # TODO 7: Sum over m in 0..n_ov-1 and i in 0..n_occ-1 for the hole term, plus a in n_occ..n_mo-1 for the particle term.
    raise NotImplementedError("Complete TODO 7: evaluate_g0w0_self_energy_diagonal")


def run_g0w0_quasiparticle_solver(
    eri_mo: np.ndarray, eps_hf: np.ndarray, n_occ: int
) -> dict[str, np.ndarray | float]:
    '''
    TODO 8: Run the complete G0W0 calculation using TODOs 4, 5, 6, and 7 for all orbitals p = 0..n_mo-1.
    Return a dict with keys:
        'eps_hf': np.ndarray (n_mo,)
        'eps_g0w0': np.ndarray (n_mo,)
        'z_weights': np.ndarray (n_mo,)
        'gap_hf_ev': float
        'gap_g0w0_ev': float
        'omega_rpa': np.ndarray
        'W_m_pq': np.ndarray
    '''
    # TODO 8: Call build_and_solve_drpa and compute_rpa_transition_potentials, then solve the linearized
    #         quasiparticle equation (TODO 4) for every orbital p = 0..n_mo-1 and compute the HOMO-LUMO gaps in eV.
    raise NotImplementedError("Complete TODO 8: run_g0w0_quasiparticle_solver")

## Part 3: The Bethe–Salpeter Equation (`BSE`) — Static Screened Coulomb Attraction, Exciton Binding Energy & Optical Spectrum (TODOs 9–12)

### Mathematical Specifications
1. **Static Screened Coulomb Interaction Tensor $W_{pq,rs}(\omega=0)$ (`compute_static_screened_coulomb` — TODO 9)**:
   Using the RPA transition potentials $W_{pq}^m$ and frequencies $\Omega_m^{\text{RPA}}$ from Part 2, the static ($\omega = 0$) screened Coulomb interaction in the MO basis is:
   $$W_{pq, rs}(\omega = 0) = (pq|rs) - 4\sum_{m=1}^{N_{ov}} \frac{W_{pq}^m\,W_{rs}^m}{\Omega_m^{\text{RPA}}}$$
   Notice that the polarization sum $-4\sum_m \frac{(W_{pp}^m)^2}{\Omega_m^{\text{RPA}}} < 0$ is strictly negative on the diagonal, meaning dielectric screening **weakens** the bare Coulomb interaction $W_{pp,pp}(0) < (pp|pp)$!

2. **Tamm–Dancoff (`TDA-BSE`) & Full Casida `BSE` Exciton Solver (`build_and_solve_bse` — TODO 10)**:
   In the singlet electron–hole basis $|ia\rangle$ ($i \in \text{occ}, a \in \text{virt}$), the `BSE` Hamiltonian replaces bare HF orbital energies with the **$G_0W_0$ quasiparticle energies $\varepsilon_p^{G_0W_0}$** and replaces bare exchange in the direct electron–hole attraction kernel with the **statically screened Coulomb interaction $W_{ij,ab}(0)$**:
   $$A_{ia, jb}^{\text{BSE}} = \left(\varepsilon_a^{G_0W_0} - \varepsilon_i^{G_0W_0}\right)\delta_{ij}\delta_{ab} + 2(ia|jb) - W_{ij, ab}(\omega = 0)$$
   $$B_{ia, jb}^{\text{BSE}} = 2(ia|jb) - W_{ib, aj}(\omega = 0)$$
   - Diagonalizing $\mathbf{A}^{\text{BSE}}\mathbf{X}_S^{\text{TDA}} = \Omega_S^{\text{TDA-BSE}}\mathbf{X}_S^{\text{TDA}}$ gives the **Tamm–Dancoff `BSE`** excitation energies.
   - Solving the full Casida equation via $\mathbf{M}^{\text{BSE}} = (\mathbf{A}^{\text{BSE}} - \mathbf{B}^{\text{BSE}})^{1/2}(\mathbf{A}^{\text{BSE}} + \mathbf{B}^{\text{BSE}})(\mathbf{A}^{\text{BSE}} - \mathbf{B}^{\text{BSE}})^{1/2}$ gives the full `BSE` optical excitation energies $\Omega_S^{\text{BSE}}$ and transition densities $(\mathbf{X}_S + \mathbf{Y}_S)_{ia}$!
   - The **Exciton Binding Energy** of the lowest optical transition $S=1$ is:
     $$E_{\text{bind}}^{\text{exciton}} = \left(\varepsilon_{\text{LUMO}}^{G_0W_0} - \varepsilon_{\text{HOMO}}^{G_0W_0}\right) - \Omega_1^{\text{BSE}}$$

3. **Oscillator Strengths $f_S$ & Optical Absorption Spectrum (`compute_bse_oscillator_strengths` & `compute_optical_absorption_spectrum` — TODOs 11–12)**:
   Given the MO transition dipole moments $\boldsymbol{\mu}_{ia}^{\text{MO}}$ of shape `(3, n_occ, n_virt)`, the transition dipole vector of exciton state $S$ is:
   $$\mathbf{d}_S = \sqrt{2}\sum_{i \in \text{occ}}\sum_{a \in \text{virt}} \boldsymbol{\mu}_{ia}^{\text{MO}}\,(X + Y)_{ia}^S, \qquad f_S = \frac{2}{3}\,\Omega_S^{\text{BSE}}\,\|\mathbf{d}_S\|_2^2$$

In [ ]:
def compute_static_screened_coulomb(
    eri_mo: np.ndarray, omega_rpa: np.ndarray, W_m_pq: np.ndarray
) -> np.ndarray:
    '''
    TODO 9: Compute the 4-index static screened Coulomb tensor W_stat[p, q, r, s] = (pq|rs) - 4 * sum_m W_m_pq[m,p,q] * W_m_pq[m,r,s] / omega_rpa[m]
    of shape (n_mo, n_mo, n_mo, n_mo).
    '''
    # TODO 9: Contract W_m_pq with itself weighted by 1.0 / omega_rpa via np.einsum and subtract 4.0 * screened_corr from eri_mo.
    raise NotImplementedError("Complete TODO 9: compute_static_screened_coulomb")


def build_and_solve_bse(
    eri_mo: np.ndarray,
    W_stat: np.ndarray,
    eps_qp: np.ndarray,
    n_occ: int,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 10: Construct the singlet A_bse and B_bse matrices of shape (n_ov, n_ov),
    solve both the Tamm-Dancoff Approximation (eigh(A_bse)) and the full Casida BSE equation,
    and compute the lowest exciton binding energy in eV:
    `exciton_binding_ev = ((eps_qp[n_occ] - eps_qp[n_occ - 1]) - omega_bse[0]) * HARTREE_TO_EV`.
    Return a dict with keys:
        'A_bse', 'B_bse', 'omega_tda_eh', 'omega_bse_eh', 'x_plus_y_bse' (shape n_occ, n_virt, n_ov),
        'qp_gap_ev', 'optical_gap_bse_ev', 'exciton_binding_ev'.
    '''
    # TODO 10: Assemble A_bse and B_bse in the (i, a) composite basis, solve TDA and full Casida BSE
    #          (using scipy.linalg.sqrtm or eigh on A - B), and compute the gaps and exciton binding energy.
    raise NotImplementedError("Complete TODO 10: build_and_solve_bse")


def compute_bse_oscillator_strengths(
    ao_dip: np.ndarray,
    C_mo: np.ndarray,
    omega_bse_eh: np.ndarray,
    x_plus_y_bse: np.ndarray,
    n_occ: int,
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 11: Transform `ao_dip` (shape 3, n_ao, n_ao) to the occupied-virtual MO block
    `dip_ia = C_occ^T @ ao_dip @ C_virt` (shape 3, n_occ, n_virt), compute the exciton transition
    dipole vectors `d_vec` (shape n_ov, 3) = sqrt(2) * sum_{ia} dip_ia * (X+Y)_{ia}^S,
    and evaluate oscillator strengths `osc_strengths = (2/3) * omega_bse_eh * sum(d_vec^2, axis=1)`.
    Return (osc_strengths, d_vec).
    '''
    # TODO 11: Transform ao_dip to dip_ia via einsum, contract with x_plus_y_bse, and compute osc_strengths.
    raise NotImplementedError("Complete TODO 11: compute_bse_oscillator_strengths")


def compute_optical_absorption_spectrum(
    energy_grid_ev: np.ndarray,
    excitation_energies_eh: np.ndarray,
    osc_strengths: np.ndarray,
    sigma_ev: float = 0.25,
) -> np.ndarray:
    '''
    TODO 12: Evaluate the Gaussian-broadened optical absorption cross-section profile S(E) on `energy_grid_ev`
    by summing `osc_strengths[S] * Gaussian(E - omega_S_ev, sigma_ev)`.
    '''
    # TODO 12: Convert excitation_energies_eh to eV and sum Gaussian peaks weighted by osc_strengths.
    raise NotImplementedError("Complete TODO 12: compute_optical_absorption_spectrum")

## Part 4: Dynamical Mean-Field Theory (`DMFT`) — Bethe Lattice Self-Consistency, Hubbard Bands & the Mott Metal–Insulator Transition (TODOs 13–16)

### Mathematical Specifications
Consider the single-band Hubbard model $\hat{H} = -t\sum_{\langle ij\rangle\sigma}\hat{c}_{i\sigma}^\dagger\hat{c}_{j\sigma} + U\sum_i \hat{n}_{i\uparrow}\hat{n}_{i\downarrow}$ at half-filling ($\langle \hat{n}_{i\sigma}\rangle = \frac{1}{2}$, chemical potential $\mu = \frac{U}{2}$) on the infinite-coordination Bethe lattice with semicircular non-interacting Density of States (half-bandwidth $D = 2t = 1$):

$$D_0(\epsilon) = \frac{2}{\pi D^2}\sqrt{D^2 - \epsilon^2}, \qquad |\epsilon| \le D$$

1. **Bethe Lattice Hilbert Transform & Algebraic `DMFT` Cavity Relation (`bethe_lattice_local_greens_function` — TODO 13)**:
   For any complex frequency $z = \omega + i\eta$ and local self-energy $\Sigma_{\text{loc}}(\omega)$ (measured relative to the Hartree shift $U/2$ at half-filling), integrating over the semicircular DOS yields the local lattice Green's function:
   $$G_{\text{loc}}(\omega) = \int_{-D}^{+D} \frac{D_0(\epsilon)\,d\epsilon}{\omega + i\eta - \epsilon - \Sigma_{\text{loc}}(\omega)}$$
   On the Bethe lattice, the cavity hybridization function $\Delta(\omega)$ and Weiss field $\mathcal{G}_0(\omega)$ satisfy the exact algebraic relation:
   $$\Delta(\omega) = \frac{D^2}{4}\,G_{\text{loc}}(\omega), \qquad \mathcal{G}_0(\omega) = \frac{1}{\omega + i\eta - \Delta(\omega)}$$

2. **Two-Site / Interpolating Impurity Self-Energy Solver (`solve_dmft_impurity_self_energy` — TODO 14)**:
   In the particle-hole symmetric two-site / pole-interpolating DMFT impurity solver (which becomes exact at both $U \to 0$ and the atomic limit $U \to \infty$ while capturing the Brinkman–Rice / Mott metal–insulator transition at $U_c \approx 1.5\,D$ to $3\,D$!), the local correlation self-energy $\Sigma_{\text{loc}}(\omega)$ is governed by the impurity hybridization weight $V_{\text{hyb}}^2 = \text{clip}\!\left(\frac{D^2}{4}\left(1 - \left(\frac{U}{U_c}\right)^2\right),\ 0,\ \frac{D^2}{4}\right)$ (for the metallic quasiparticle Kondo bath mode at $\epsilon_{\text{bath}} = 0$) plus the Hubbard atomic-limit pole:
   $$\Sigma_{\text{loc}}(\omega) = \frac{U^2}{4}\frac{1}{\omega + i\eta - \frac{9 V_{\text{hyb}}^2}{\omega + i\eta}}$$
   Look at the two limits of this exact analytical form:
   - **Metallic Fermi Liquid ($U < U_c$, $V_{\text{hyb}} > 0$)**: As $\omega \to 0$, $\Sigma_{\text{loc}}(\omega) \to -\frac{U^2}{36 V_{\text{hyb}}^2}\,\omega$, giving a finite negative slope $\frac{\partial\text{Re}\Sigma}{\partial\omega}\big|_{\omega=0} = -\frac{U^2}{36 V_{\text{hyb}}^2} < 0$, a coherent **central Kondo quasiparticle peak** at $\omega = 0$ with quasiparticle weight $Z(U) = \left(1 + \frac{U^2}{36 V_{\text{hyb}}^2}\right)^{-1} = 1 - \left(\frac{U}{U_c}\right)^2$ ( when $U_c = \frac{3}{\sqrt{2}}D$), AND **Lower and Upper Hubbard Bands** at $\omega \approx \pm\sqrt{\frac{U^2}{4} + 9 V_{\text{hyb}}^2}$!
   - **Mott Insulator ($U \ge U_c$, $V_{\text{hyb}} = 0$)**: The central quasiparticle peak vanishes ($Z(U) = 0$) and $\Sigma_{\text{loc}}(\omega) = \frac{U^2}{4(\omega + i\eta)}$ develops a pole at $\omega = 0$ that splits the band into two insulating **Hubbard bands at $\omega = \pm U/2$**!

3. **Self-Consistent `DMFT` Iteration Loop (`run_dmft_self_consistent_loop` — TODO 15)** & **Mott Metal–Insulator Phase Diagram Scan (`scan_dmft_mott_transition` — TODO 16)**:
   Iterate $\Sigma_{\text{loc}}(\omega) \to G_{\text{loc}}(\omega) \to \Delta(\omega) \to V_{\text{hyb}}^2 \to \Sigma_{\text{loc}}^{\text{new}}(\omega)$ to self-consistency and track the three-peak spectral function $A(\omega) = -\frac{1}{\pi}\text{Im}\,G_{\text{loc}}(\omega)$, zero-frequency density of states $A(0)$, and quasiparticle weight $Z(U)$ across the Mott transition!

In [ ]:
def bethe_lattice_local_greens_function(
    omega_grid: np.ndarray,
    Sigma_loc: np.ndarray,
    half_bandwidth_D: float = 1.0,
    eta: float = 0.03,
    n_eps_quad: int = 201,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 13: Numerically integrate the semicircular Bethe lattice DOS D_0(eps) = (2 / (pi * D^2)) * sqrt(D^2 - eps^2)
    over `n_eps_quad` grid points in [-D, +D] (using `np.trapz` or `scipy.integrate.trapezoid`) to compute:
    - `G_loc` of shape (len(omega_grid),)
    - `Delta_hyb = (half_bandwidth_D**2 / 4.0) * G_loc`
    - `A_loc = -(1.0 / np.pi) * G_loc.imag`
    Return (G_loc, Delta_hyb, A_loc).
    '''
    # TODO 13: Build eps_grid in [-D, +D], D_0(eps_grid), integrate D_0(eps) / (omega + 1j*eta - eps - Sigma_loc),
    #          and return (G_loc, Delta_hyb, A_loc).
    raise NotImplementedError("Complete TODO 13: bethe_lattice_local_greens_function")


def evaluate_two_site_dmft_self_energy(
    omega_grid: np.ndarray,
    u_hubbard: float,
    v_hyb_sq: float,
    eta: float = 0.03,
) -> tuple[np.ndarray, float]:
    '''
    TODO 14: Evaluate the two-site / pole-interpolating DMFT impurity self-energy on `omega_grid`:
    z = omega_grid + 1j * eta
    Sigma_loc(omega) = 0.25 * (u_hubbard**2) / (z - 9.0 * v_hyb_sq / z)
    and compute the analytical quasiparticle renormalization weight:
    z_qp = 1.0 / (1.0 + (u_hubbard**2) / (36.0 * v_hyb_sq)) if v_hyb_sq > 1e-12 else 0.0.
    Return (Sigma_loc, z_qp).
    '''
    # TODO 14: Compute Sigma_loc on omega_grid and z_qp, and return (Sigma_loc, z_qp).
    raise NotImplementedError("Complete TODO 14: evaluate_two_site_dmft_self_energy")


def run_dmft_self_consistent_loop(
    omega_grid: np.ndarray,
    u_hubbard: float,
    half_bandwidth_D: float = 1.0,
    eta: float = 0.03,
    max_iter: int = 80,
    tol: float = 1e-7,
    mix: float = 0.5,
) -> dict[str, np.ndarray | float | int | bool]:
    '''
    TODO 15: Run the self-consistent two-site DMFT loop at interaction `u_hubbard`:
    In the two-site DMFT self-consistency condition on the Bethe lattice (Potthoff, PRB 64, 165114 (2001)),
    the bath hybridization weight updates at each step via `v_hyb_sq_new = 0.25 * (half_bandwidth_D**2) * z_qp`,
    where `z_qp` is the quasiparticle weight from `evaluate_two_site_dmft_self_energy`.
    Start from `v_hyb_sq = 0.25 * (half_bandwidth_D**2)` and iterate:
    1. Evaluate (Sigma_loc, z_qp) via TODO 14.
    2. Update v_hyb_sq_next = mix * (0.25 * half_bandwidth_D**2 * z_qp) + (1 - mix) * v_hyb_sq.
    3. Stop when |v_hyb_sq_next - v_hyb_sq| < tol (if v_hyb_sq < 1e-5, set v_hyb_sq = 0.0 for the Mott insulator).
    Finally, evaluate (G_loc, Delta_hyb, A_loc) via TODO 13.
    Return a dict with keys:
    'converged', 'u_hubbard', 'z_qp', 'v_hyb_sq', 'G_loc', 'Delta_hyb', 'A_loc', 'Sigma_loc', 'n_iterations'.
    '''
    # TODO 15: Iterate the self-consistent DMFT loop and return all 9 quantities.
    raise NotImplementedError("Complete TODO 15: run_dmft_self_consistent_loop")


def scan_dmft_mott_transition(
    u_grid: np.ndarray, half_bandwidth_D: float = 1.0
) -> dict[str, np.ndarray | float]:
    '''
    TODO 16: Across `u_grid`, run `run_dmft_self_consistent_loop` on `omega_grid = np.linspace(-3.5, 3.5, 301)`
    and record the quasiparticle weight `z_qp_curve` and zero-frequency spectral density `a_zero_curve` (A_loc at omega=0).
    Also return the analytical critical Mott interaction `u_crit = 3.0 * half_bandwidth_D / np.sqrt(2.0)` (~2.1213 * D).
    Return a dict with keys 'u_grid', 'z_qp_curve', 'a_zero_curve', 'u_crit'.
    '''
    # TODO 16: Loop over u_grid, run TODO 15, extract z_qp and A_loc[len(omega_grid)//2], and return.
    raise NotImplementedError("Complete TODO 16: scan_dmft_mott_transition")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section05g_greens_checkpoints()` below as you implement each `# TODO` function to test your non-interacting Green's functions, Dyson's equation, `GF2` self-energy, `dRPA` Casida solver, $G_0W_0$ quasiparticle solver, `BSE` exciton solver and optical spectrum, and `DMFT` Mott metal–insulator transition simulator!

In [2]:
def run_section05g_greens_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 16 #TODO functions in Section 05g (GF, GW, BSE & DMFT).'''
    passed = 0
    total_todos = 16

    try:
        # --- PART 1 CHECKS: GREEN'S FUNCTIONS, DYSON & GF2 (TODOs 1-4) ---
        w_test = np.linspace(-25.0, 5.0, 4000)
        G0_d, A0_w = evaluate_noninteracting_greens_function(w_test, eps_hf_h2o, eta=0.04)
        dw = w_test[1] - w_test[0]
        assert G0_d.shape == (4000, 7) and np.isclose(np.sum(A0_w) * dw, 7.0, atol=0.05)
        passed += 1
        print(f"[PASS] TODO 1 : evaluate_noninteracting_greens_function (Integrated spectral weight = {np.sum(A0_w)*dw:.3f} / 7.0)")

        G_dys, A_dys = solve_dyson_equation(-0.39, eps_hf_h2o, np.zeros((7, 7)), eta=0.02)
        assert G_dys.shape == (7, 7) and A_dys > 1.0
        passed += 1
        print("[PASS] TODO 2 : solve_dyson_equation")

        sig_gf2_homo = evaluate_gf2_self_energy_diagonal(eps_hf_h2o[4], 4, eri_mo_h2o, eps_hf_h2o, n_occ=5)
        assert isinstance(sig_gf2_homo, complex) and abs(sig_gf2_homo.real) > 1e-4
        passed += 1
        print(f"[PASS] TODO 3 : evaluate_gf2_self_energy_diagonal (Re Sigma_HOMO^(2) = {sig_gf2_homo.real:+.6f} Eh)")

        qp_gf2 = solve_linearized_quasiparticle_equation(
            4, eps_hf_h2o,
            lambda w, p: evaluate_gf2_self_energy_diagonal(w, p, eri_mo_h2o, eps_hf_h2o, n_occ=5)
        )
        assert 0.80 < qp_gf2["z_weight"] <= 1.0
        passed += 1
        print(f"[PASS] TODO 4 : solve_linearized_quasiparticle_equation (GF2 HOMO Z = {qp_gf2['z_weight']:.4f}, E_QP = {qp_gf2['eps_qp_lin']:+.5f} Eh)")

        # --- PART 2 CHECKS: dRPA & G0W0 QUASIPARTICLE SOLVER (TODOs 5-8) ---
        drpa_h2o = build_and_solve_drpa(eri_mo_h2o, eps_hf_h2o, n_occ=5)
        assert drpa_h2o["omega_rpa"].shape == (10,) and np.all(drpa_h2o["omega_rpa"] > 0.0)
        assert drpa_h2o["x_plus_y"].shape == (5, 2, 10)
        passed += 1
        print(f"[PASS] TODO 5 : build_and_solve_drpa (Lowest dRPA mode = {drpa_h2o['omega_rpa'][0]*HARTREE_TO_EV:.2f} eV)")

        W_m_pq = compute_rpa_transition_potentials(eri_mo_h2o, drpa_h2o["x_plus_y"], n_occ=5)
        assert W_m_pq.shape == (10, 7, 7) and np.allclose(W_m_pq, W_m_pq.transpose(0, 2, 1))
        passed += 1
        print("[PASS] TODO 6 : compute_rpa_transition_potentials")

        sig_gw_homo = evaluate_g0w0_self_energy_diagonal(
            eps_hf_h2o[4], 4, eps_hf_h2o, drpa_h2o["omega_rpa"], W_m_pq, n_occ=5
        )
        assert abs(sig_gw_homo.real) > 1e-4
        passed += 1
        print(f"[PASS] TODO 7 : evaluate_g0w0_self_energy_diagonal (Re Sigma_HOMO^G0W0 = {sig_gw_homo.real:+.6f} Eh)")

        gw_res = run_g0w0_quasiparticle_solver(eri_mo_h2o, eps_hf_h2o, n_occ=5)
        assert gw_res["gap_g0w0_ev"] < gw_res["gap_hf_ev"] and np.all((gw_res["z_weights"] > 0.7) & (gw_res["z_weights"] <= 1.0))
        passed += 1
        print(f"[PASS] TODO 8 : run_g0w0_quasiparticle_solver (HF Gap = {gw_res['gap_hf_ev']:.2f} eV -> G0W0 Gap = {gw_res['gap_g0w0_ev']:.2f} eV)")

        # --- PART 3 CHECKS: BETHE-SALPETER EQUATION (BSE) & EXCITONS (TODOs 9-12) ---
        W_stat = compute_static_screened_coulomb(eri_mo_h2o, drpa_h2o["omega_rpa"], W_m_pq)
        assert W_stat.shape == (7, 7, 7, 7) and W_stat[4, 4, 4, 4] < eri_mo_h2o[4, 4, 4, 4]
        passed += 1
        print(f"[PASS] TODO 9 : compute_static_screened_coulomb (Bare (44|44) = {eri_mo_h2o[4,4,4,4]:.4f} -> Screened W_4444(0) = {W_stat[4,4,4,4]:.4f} Eh)")

        bse_res = build_and_solve_bse(eri_mo_h2o, W_stat, gw_res["eps_g0w0"], n_occ=5)
        assert bse_res["exciton_binding_ev"] > 5.0 and bse_res["optical_gap_bse_ev"] < bse_res["qp_gap_ev"]
        passed += 1
        print(f"[PASS] TODO 10: build_and_solve_bse (G0W0 QP Gap = {bse_res['qp_gap_ev']:.2f} eV, BSE Optical Gap = {bse_res['optical_gap_bse_ev']:.2f} eV, Exciton Binding = {bse_res['exciton_binding_ev']:.2f} eV)")

        f_osc, d_vec = compute_bse_oscillator_strengths(
            ao_dip_h2o, C_hf_h2o, bse_res["omega_bse_eh"], bse_res["x_plus_y_bse"], n_occ=5
        )
        assert f_osc.shape == (10,) and np.all(f_osc >= 0.0) and np.max(f_osc) > 0.01
        passed += 1
        print(f"[PASS] TODO 11: compute_bse_oscillator_strengths (Max oscillator strength f = {np.max(f_osc):.4f})")

        e_uv = np.linspace(5.0, 30.0, 300)
        abs_spec = compute_optical_absorption_spectrum(e_uv, bse_res["omega_bse_eh"], f_osc, sigma_ev=0.4)
        assert abs_spec.shape == (300,) and np.max(abs_spec) > 0.0
        passed += 1
        print("[PASS] TODO 12: compute_optical_absorption_spectrum")

        # --- PART 4 CHECKS: DYNAMICAL MEAN-FIELD THEORY (DMFT) & MOTT TRANSITION (TODOs 13-16) ---
        w_dmft = np.linspace(-3.5, 3.5, 301)
        G_b0, Del_b0, A_b0 = bethe_lattice_local_greens_function(w_dmft, np.zeros_like(w_dmft, dtype=np.complex128), half_bandwidth_D=1.0)
        dw_d = w_dmft[1] - w_dmft[0]
        assert np.isclose(np.sum(A_b0) * dw_d, 1.0, atol=0.02) and np.isclose(A_b0[150], 2.0 / np.pi, atol=0.03)
        passed += 1
        print(f"[PASS] TODO 13: bethe_lattice_local_greens_function (Non-interacting A(0) = {A_b0[150]:.4f} vs 2/pi = {2/np.pi:.4f})")

        sig_imp, z_imp = evaluate_two_site_dmft_self_energy(w_dmft, u_hubbard=1.2, v_hyb_sq=0.18)
        assert sig_imp.shape == (301,) and 0.0 < z_imp < 1.0
        passed += 1
        print(f"[PASS] TODO 14: evaluate_two_site_dmft_self_energy (Z_imp = {z_imp:.4f})")

        dmft_metal = run_dmft_self_consistent_loop(w_dmft, u_hubbard=1.2, half_bandwidth_D=1.0)
        dmft_mott = run_dmft_self_consistent_loop(w_dmft, u_hubbard=2.6, half_bandwidth_D=1.0)
        assert dmft_metal["z_qp"] > 0.40 and dmft_mott["z_qp"] == 0.0
        assert dmft_metal["A_loc"][150] > 0.50 and dmft_mott["A_loc"][150] < 0.05
        passed += 1
        print(f"[PASS] TODO 15: run_dmft_self_consistent_loop (Metal U=1.2 Z={dmft_metal['z_qp']:.3f} vs Mott Insulator U=2.6 Z={dmft_mott['z_qp']:.3f})")

        mott_scan = scan_dmft_mott_transition(np.linspace(0.0, 2.8, 15), half_bandwidth_D=1.0)
        assert np.isclose(mott_scan["z_qp_curve"][0], 1.0) and mott_scan["z_qp_curve"][-1] == 0.0
        assert np.isclose(mott_scan["u_crit"], 3.0 / np.sqrt(2.0))
        passed += 1
        print(f"[PASS] TODO 16: scan_dmft_mott_transition (Brinkman-Rice / Mott critical U_c = {mott_scan['u_crit']:.4f} D)")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 05g (Green's Functions, GW, BSE & DMFT) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section05g_greens_checkpoints()
print("Section 05g Green's Function checkpoint suite loaded. Implement TODOs 1-16 and call `run_section05g_greens_checkpoints()`!")

Section 05g Green's Function checkpoint suite loaded. Implement TODOs 1-16 and call `run_section05g_greens_checkpoints()`!
